# LangChain Agents Intro

- LangChain is one of the most popular open source libraries for AI Engineers.

- It's goal is to abstract away the complexity in building AI software, provide easy-to-use building blocks, and make it easier when switching between AI service providers.

- In this example, we will introduce LangChain's Agents, adding the ability to use tools such as search and calculators to complete tasks that normal LLMs cannot fufil.


In [1]:
import json
import os
import warnings
from pathlib import Path
from typing import Any, Generator, Iterable, Type, TypeVar

# Standard imports
import numpy as np
import pandas as pd
import polars as pl

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

In [2]:
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # Bright green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)

In [3]:
go_up_from_current_directory(go_up=2)

from settings import refresh_settings  # noqa: E402

settings = refresh_settings()

/Users/mac/Desktop/Projects/RAG-Tutorials


In [4]:
from langchain_openai import ChatOpenAI

model_str_remote: str = "google/gemini-2.0-flash-001"
model_str_local: str = "llama3.1:8b"  # llama3.1:8b, gemma3n:e4b, llama3.2:3b

# Deterministic responses
remote_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),  # type: ignore
    base_url=settings.OPENROUTER_URL,
    temperature=0.0,
    model=model_str_remote,
)

local_llm = ChatOpenAI(
    api_key=settings.OLLAMA_API_KEY.get_secret_value(),
    base_url=settings.OLLAMA_URL,
    temperature=0.0,
    model=model_str_local,
)

<br>

## Introduction To Tools

- Tools are a way augment our LLMs with code execution. A tool is simply a function formatted so that our agent can undertstand how to use it, and then execute it.

- Let's start by creating a few simple tools.

- We can use the @tool decorator to create an LLM-compatible tool from a standard python function — this function should include a few things for optimal performance:

    - A docstring describing what the tool does and when it should be used, this will be read by our LLM/agent and used to decide when to use the tool, and also how to use the tool.

    - Clear parameter names that ideally tell the LLM what each parameter is, if it isn't clear we make sure the docstring explains what the parameter is for and how to use it.

    - Both parameter and return type annotations.

In [5]:
from langchain_core.tools import tool


@tool
def add(x: float, y: float) -> float:
    """Add two numbers together."""
    return x + y


@tool
def multiply(x: float, y: float) -> float:
    """Multiply two numbers together."""
    return x * y


@tool
def exponentiate(x: float, y: float) -> float:
    """Raise x to the power of y."""
    return x**y


@tool
def subtract(x: float, y: float) -> float:
    """Subtract y from x."""
    return x - y

- With the `@tool` decorator our function is turned into a StructuredTool object, which we can see below:

In [6]:
console.print(add)

StructuredTool(
    name='add',
    description='Add two numbers together.',
    args_schema=<class 'langchain_core.utils.pydantic.add'>,
    func=<function add at 0x12a47d580>
)

In [7]:
print(f"{add.name=}\n{add.description=}")

add.name='add'
add.description='Add two numbers together.'


In [8]:
console.print(add.args_schema.model_json_schema())

{
    'description': 'Add two numbers together.',
    'properties': {'x': {'title': 'X', 'type': 'number'}, 'y': {'title': 'Y', 'type': 'number'}},
    'required': ['x', 'y'],
    'title': 'add',
    'type': 'object'
}

- When invoking the tool, a `JSON` string output by the LLM will be parsed into `JSON` and then consumed as `kwargs`, similar to the below:


In [9]:
llm_output_string = '{"x": 5, "y": 2}'  # example llm output string
llm_output_dict = json.loads(llm_output_string)  # Convert JSON string to Python dict
llm_output_dict

{'x': 5, 'y': 2}

- This is then passed into the tool function as kwargs (keyword arguments) as indicated by the `** operator` - the `** operator` is used to unpack the dictionary into keyword arguments.

In [10]:
exponentiate.func(**llm_output_dict)  # Unpack the dictionary into keyword arguments

25

## Creating An Agent

- We're going to construct a simple tool calling agent using `LangChain Epression Language (LCEL)` to construct the agent. 

- We will cover LCEL more in the next chapter, but for now - all we need to know is that our agent will be constructed using syntax and components like so:

    ```py
    agent = (
        <input parameters, including chat history and user query>
        | <prompt>
        | <LLM with tools>
    )
    ```

- We need this agent to remember previous interactions within the conversation.

- To do that, we will use the ChatPromptTemplate with a system message, a placeholder for our chat history, a placeholder for the user query, and finally a placeholder for the agent scratchpad.

- The agent scratchpad is where the agent will write it's "notes" as it is working through multiple internal thought and tool-use steps to produce a final output to the user.

In [11]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

llm = local_llm  # or remote_llm

In [12]:
from langchain_core.runnables.base import RunnableSerializable

tools = [add, multiply, exponentiate, subtract]

# Agent runnable
agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", []),
    }
    | prompt
    | llm.bind_tools(tools, tool_choice="any")
)

In [13]:
tool_call = agent.invoke({"input": "What is the sum of 5 and -2?", "chat_history": []})

console.print(tool_call)

AIMessage(
    content='',
    additional_kwargs={
        'tool_calls': [
            {
                'id': 'call_przaz77n',
                'function': {'arguments': '{"x":5,"y":-2}', 'name': 'add'},
                'type': 'function',
                'index': 0
            }
        ],
        'refusal': None
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 22,
            'prompt_tokens': 341,
            'total_tokens': 363,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'llama3.1:8b',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-186',
        'service_tier': None,
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='run--f9660367-2fe1-4a8f-a7be-0cdc954108a2-0',
    tool_calls=[{'name': 'add', 'args': {'x': 5, 'y': -2}, 'id': 'call_przaz77n', 'type': 'tool_call'}],
    usage_metadata={
        'input_tokens': 341,
        'output_tokens': 22,
        'total_tokens': 363,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [14]:
tool_call.tool_calls

[{'name': 'add',
  'args': {'x': 5, 'y': -2},
  'id': 'call_przaz77n',
  'type': 'tool_call'}]

- From here, we have the tool name that our LLM wants to use and the args that it wants to pass to that tool.

- We can see that the tool add is being used with the arguments `x=5` and `y=-2`. 

- The `agent.invoke` method has not executed the tool function; we need to write that part of the agent code ourselves.

- Executing the tool code requires two steps:
    - Map the tool name to the tool function.
    - Execute the tool function with the generated args.

In [15]:
# Create a mapping of tool names to tool functions
name2tool = {tool.name: tool.func for tool in tools}
console.print(name2tool)

{
    'add': <function add at 0x12a47d580>,
    'multiply': <function multiply at 0x12a4c36a0>,
    'exponentiate': <function exponentiate at 0x12a4e31a0>,
    'subtract': <function subtract at 0x12a4e37e0>
}

In [16]:
tool_name: str = tool_call.tool_calls[0]["name"]
tool_name

'add'

In [ ]:
# === Function ===
# tool_call.tool_calls[0][tool_name]

# === Args ===
# tool_call.tool_calls[0]["args"]

# Call the tool function with the arguments from the tool call
name2tool[tool_name](**tool_call.tool_calls[0]["args"])

3

In [ ]:
# Execute the tool function with the generated args
tool_exec_content = name2tool[tool_call.tool_calls[0]["name"]](**tool_call.tool_calls[0]["args"])
tool_exec_content

3

- That is our answer and tool execution logic. 

- We feed this back into our LLM via the agent_scratchpad placeholder.


In [ ]:
from langchain_core.messages import ToolMessage

tool_exec = ToolMessage(
    content=f"The {tool_call.tool_calls[0]['name']} tool returned {tool_exec_content}.",
    tool_call_id=tool_call.tool_calls[0]["id"],
)

out = agent.invoke(
    {
        "input": "What is the sum of 5 and -2?",
        "chat_history": [],
        "agent_scratchpad": [tool_call, tool_exec],  # new
    }
)

console.print(out)

AIMessage(
    content='The sum of 5 and -2 is 3.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 13,
            'prompt_tokens': 110,
            'total_tokens': 123,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'llama3.1:8b',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-664',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--b250dd98-405a-45e2-830c-c754b3f05637-0',
    usage_metadata={
        'input_tokens': 110,
        'output_tokens': 13,
        'total_tokens': 123,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [20]:
tool_exec

ToolMessage(content='The add tool returned 3.', tool_call_id='call_przaz77n')

### Add Structure To The Output

- It removes the possibility of an agent using the direct content field when it is not appropriate; for example, some LLMs (particularly smaller ones) may try to use the content field when using a tool.

- We can enforce a specific structured output in our answers. 

- Structured outputs are handy when we require particular fields for downstream code or multi-part answers. 
    - For example, a RAG agent may return a natural language answer and a list of sources used to generate that answer.


In [21]:
@tool
def final_answer(answer: str, tools_used: list[str]) -> str:
    """Use this tool to provide a final answer to the user.
    The answer should be in natural language as this will be provided
    to the user directly. The tools_used must include a list of tool
    names that were used within the `scratchpad`.
    """
    return {"answer": answer, "tools_used": tools_used}

In [22]:
tools = [add, multiply, exponentiate, subtract, final_answer]
name2tool = {tool.name: tool.func for tool in tools}

agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", []),
    }
    | prompt
    | llm.bind_tools(tools, tool_choice="any")
)

tool_call = agent.invoke({"input": "What is the sum of 5 and -2?", "chat_history": []})

console.print(tool_call.tool_calls)

[{'name': 'add', 'args': {'x': 5, 'y': -2}, 'id': 'call_1970uhh3', 'type': 'tool_call'}]

In [23]:
tool_out = name2tool[tool_call.tool_calls[0]["name"]](**tool_call.tool_calls[0]["args"])

tool_exec = ToolMessage(
    content=f"The {tool_call.tool_calls[0]['name']} tool returned {tool_out}",
    tool_call_id=tool_call.tool_calls[0]["id"],
)

out = agent.invoke(
    {
        "input": "What is 10 + 10",
        "chat_history": [],
        "agent_scratchpad": [tool_call, tool_exec],
    }
)

console.print(out)

AIMessage(
    content='The result of 10 + 10 is 20.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 13,
            'prompt_tokens': 105,
            'total_tokens': 118,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'llama3.1:8b',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-258',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--2189937f-d8a8-4e29-8f55-ebc522acf0c3-0',
    usage_metadata={
        'input_tokens': 105,
        'output_tokens': 13,
        'total_tokens': 118,
        'input_token_details': {},
        'output_token_details': {}
    }
)

#### Note:

- If `out.content=""`, run `out.tool_calls`. It shouldn't be empty.

In [ ]:
out.tool_calls

## Building A Custom Agent Execution Loop

- We've worked through each step of our agent code, but it doesn't run without us running every step. 

- We must write a class to handle all the logic we just worked through.

In [ ]:
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage


class CustomAgentExecutor:
    chat_history: list[BaseMessage]

    def __init__(self, max_iterations: int = 4) -> None:
        self.chat_history = []
        self.max_iterations = max_iterations
        self._create_agent()

    def _create_agent(self) -> RunnableSerializable:
        """Create the agent runnable with the tools and prompt."""
        self.agent: RunnableSerializable = (
            {
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"],
                "agent_scratchpad": lambda x: x.get("agent_scratchpad", []),
            }
            | prompt
            | llm.bind_tools(tools, tool_choice="any")
        )

    def invoke(self, input: str) -> dict:
        # invoke the agent but we do this iteratively in a loop until
        # reaching a final answer
        count = 0
        agent_scratchpad = []
        while count < self.max_iterations:
            # invoke a step for the agent to generate a tool call
            tool_call = self.agent.invoke(
                {"input": input, "chat_history": self.chat_history, "agent_scratchpad": agent_scratchpad}
            )
            # add initial tool call to scratchpad
            agent_scratchpad.append(tool_call)
            # otherwise we execute the tool and add it's output to the agent scratchpad
            tool_name = tool_call.tool_calls[0]["name"]
            tool_args = tool_call.tool_calls[0]["args"]
            tool_call_id = tool_call.tool_calls[0]["id"]
            tool_out = name2tool[tool_name](**tool_args)
            # add the tool output to the agent scratchpad
            tool_exec = ToolMessage(content=f"{tool_out}", tool_call_id=tool_call_id)
            agent_scratchpad.append(tool_exec)
            # add a print so we can see intermediate steps
            print(f"{count}: {tool_name}({tool_args})")
            count += 1
            # if the tool call is the final answer tool, we stop
            if tool_name == "final_answer":
                break
        # add the final output to the chat history
        final_answer = tool_out["answer"]
        self.chat_history.extend([HumanMessage(content=input), AIMessage(content=final_answer)])
        # return the final answer in dict form
        return json.dumps(tool_out)

In [51]:
agent_executor = CustomAgentExecutor(max_iterations=5)

# Invoke the agent executor with an input
result = agent_executor.invoke("What is the product of 5 and -2?")
console.print(result)

Iteration 0: multiply({'x': 5, 'y': -2}) = -10


{'answer': '-10', 'tools_used': ['multiply']}

In [ ]:
from langchain_core.messages import BaseMessage


class CustomAgentExecutorUpdated:
    """Agent executor that manages a conversation with tools."""

    def __init__(self, llm=None, tools=None, max_iterations: int = 4, verbose: bool = False) -> None:
        """Initialize the agent executor.

        Parameters
        ----------
        llm : ChatModel, optional
            Language model to use for the agent
        tools : list[Tool], optional
            list of tools available to the agent
        max_iterations : int, default=4
            Maximum number of tool calling iterations before stopping
        verbose : bool, default=False
            Whether to print detailed execution logs
        """
        self.chat_history: list[BaseMessage] = []
        self.max_iterations = max_iterations
        self.verbose = verbose

        # Allow tools and LLM to be passed during initialization
        self._tools = tools or []
        self._llm = llm or local_llm
        self._name2tool = {tool.name: tool.func for tool in self._tools}

        # Create the agent
        self._create_agent()

    def _create_agent(self) -> None:
        """Create the agent runnable."""
        self.agent = (
            {
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"],
                "agent_scratchpad": lambda x: x.get("agent_scratchpad", []),
            }
            | prompt
            | self._llm.bind_tools(self._tools, tool_choice="any")
        )

    def _log(self, message: str) -> None:
        """Log a message if verbose mode is enabled."""
        if self.verbose:
            print(message)

    def add_tool(self, tool) -> None:
        """Add a new tool to the agent.

        Parameters
        ----------
        tool : Tool
            Tool to add to the agent
        """
        self._tools.append(tool)
        self._name2tool[tool.name] = tool.func
        self._create_agent()  # Recreate the agent with the new tool

    def reset(self) -> None:
        """Reset the chat history."""
        self.chat_history = []

    def invoke(self, input: str) -> dict[str, Any]:
        """Invoke the agent with the given input.

        Parameters
        ----------
        input : str
            User input to the agent

        Returns
        -------
        dict[str, Any]
            Response containing the answer and tools used
        """
        count: int = 0
        agent_scratchpad: list[Any] = []
        tools_used: list[str] = []

        while count < self.max_iterations:
            # Invoke agent to make tool call
            tool_call = self.agent.invoke(
                {
                    "input": input,
                    "chat_history": self.chat_history,
                    "agent_scratchpad": agent_scratchpad,
                }
            )

            # Handle empty tool calls
            if not tool_call.tool_calls:
                if tool_call.content:
                    final_answer = tool_call.content
                    self._log(f"Direct response (no tools): {final_answer}")
                    break
                else:
                    self._log("Warning: Empty tool call and content")
                    final_answer = "I couldn't process your request properly."
                    break

            # Add tool call to scratchpad
            agent_scratchpad.append(tool_call)

            # Extract information from tool call
            tool_name = tool_call.tool_calls[0]["name"]
            tool_args = tool_call.tool_calls[0]["args"]
            tool_call_id = tool_call.tool_calls[0]["id"]

            # Track used tools
            tools_used.append(tool_name)

            try:
                # Execute the tool
                tool_output = self._name2tool[tool_name](**tool_args)
                self._log(f"Iteration {count}: {tool_name}({tool_args}) = {tool_output}")
            except KeyError:
                # Handle unknown tool
                tool_output = f"Error: Tool '{tool_name}' not found"
                self._log(f"Error: Unknown tool '{tool_name}'")
            except Exception as e:
                # Handle tool execution errors
                tool_output = f"Error executing tool: {str(e)}"
                self._log(f"Error executing {tool_name}: {str(e)}")

            # Create tool execution message
            tool_exec = ToolMessage(
                content=f"The {tool_name} tool returned {tool_output}",
                tool_call_id=tool_call_id,
            )

            # Add tool execution to scratchpad
            agent_scratchpad.append(tool_exec)
            count += 1

            # Exit if the tool name is final_answer or content is empty
            if tool_name == "final_answer" or tool_call.content == "":
                break

        # Extract final answer
        if tool_name == "final_answer":
            try:
                final_answer = tool_output["answer"]
            except (TypeError, KeyError):
                final_answer = str(tool_output)
        else:
            # If we hit max iterations without final_answer
            final_answer = str(tool_output)

        # Add final answer to chat history
        self.chat_history.extend([HumanMessage(content=input), AIMessage(content=final_answer)])

        return {"answer": final_answer, "tools_used": tools_used}

In [ ]:
# Initialize the agent executor with the tools
agent_executor = CustomAgentExecutor(
    max_iterations=5,
    tools=tools,  # Pass the tools list we defined earlier
    llm=local_llm,
    verbose=True,  # Enable logging for debugging
)

# Invoke the agent executor with an input
result = agent_executor.invoke("What is the sum of 5 and -2? Add 10 to the result")
console.print(result)

In [ ]:
agent_executor.chat_history

In [ ]:
def my_func(a: int) -> int:
    return a * 2


my_func("5")
